# Chapter 1 — Moving Average
## Case Study: FMCG Beverage — Baseline Demand Forecasting

**Reference:** Vandeput (2021), Chapter 1  
**Author:** Hanzila Bin Younus — EM CDE, University of Salzburg  

---

### Industry Scenario

> **Company:** Global energy drink manufacturer (Central European distribution hub)  
> **Problem:** Demand planning team needs the simplest explainable baseline forecast  
> for weekly SKU replenishment. Management wants to know: *what is our minimum viable*  
> *forecast before we invest in complex models?*  
> **Data:** 104 weeks of weekly demand history (2 years)  

### Formula
$$f_t = \frac{1}{n} \sum_{i=1}^{n} d_{t-i}$$

### Key Question
> *Which window size — 3, 8, or 12 weeks — minimises forecast error for weekly replenishment?*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fb',
                     'axes.spines.top': False, 'axes.spines.right': False})
C = {'demand': '#1a2332', 'naive': '#94a3b8', 'ma3': '#e07b00',
     'ma8': '#0077b6', 'ma12': '#6246ea'}

In [ ]:
# ── 1. GENERATE FMCG DEMAND ──────────────────────────────────────────
np.random.seed(42)
n_weeks = 104
t = np.arange(n_weeks)
weeks = pd.date_range('2022-01-03', periods=n_weeks, freq='W-MON')

base        = 5000 + 12 * t
seasonality = 800 * np.sin(2 * np.pi * (t - 12) / 52)
noise       = np.random.normal(0, 180, n_weeks)
promo       = np.zeros(n_weeks)
promo[38]   = 1200   # Q3 event spike
promo[90]   = 950    # year-end promo

demand = np.clip(base + seasonality + noise + promo, 0, None)
print(f'Demand — Mean: {demand.mean():.0f} | Min: {demand.min():.0f} | Max: {demand.max():.0f} | CoV: {demand.std()/demand.mean()*100:.1f}%')

In [ ]:
# ── 2. MODEL IMPLEMENTATIONS ─────────────────────────────────────────
def naive_forecast(d, extra=8):
    """Naïve — last observed value. The benchmark every model must beat."""
    cols = len(d)
    f = np.full(cols + extra, np.nan)
    for i in range(1, cols): f[i] = d[i-1]
    f[cols:] = d[cols-1]
    return f

def moving_average(d, n=3, extra=8):
    """Moving Average — average of last n periods."""
    cols = len(d)
    f = np.full(cols + extra, np.nan)
    for i in range(n, cols): f[i] = np.mean(d[i-n:i])
    f[cols:] = np.mean(d[cols-n:cols])
    return f

def compute_kpis(actual, forecast, naive_f, train_end):
    a = actual[train_end:]; f = forecast[train_end:train_end+len(a)]
    fn = naive_f[train_end:train_end+len(a)]
    mask = ~(np.isnan(a)|np.isnan(f)); maskn = ~(np.isnan(a)|np.isnan(fn))
    e = f[mask]-a[mask]; en = fn[maskn]-a[maskn]
    mae = np.abs(e).mean(); mae_n = np.abs(en).mean()
    return {'MAE': round(mae,0), 'MAE%': round(mae/a[mask].mean()*100,1),
            'Bias': round(e.mean(),0), 'RMSE': round(np.sqrt((e**2).mean()),0),
            'FVA%': round((mae_n-mae)/mae_n*100,1)}

TRAIN_END = 80; EXTRA = 8
f_naive = naive_forecast(demand, EXTRA)
f_ma3   = moving_average(demand, 3, EXTRA)
f_ma8   = moving_average(demand, 8, EXTRA)
f_ma12  = moving_average(demand, 12, EXTRA)
print('Models computed')

In [ ]:
# ── 3. MAIN VISUALISATION ────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                          gridspec_kw={'height_ratios': [3, 1]})
ax = axes[0]
x = np.arange(len(demand) + EXTRA)

ax.axvspan(0, TRAIN_END, alpha=0.06, color='#0077b6')
ax.axvspan(TRAIN_END, len(demand), alpha=0.06, color='#6246ea')
ax.axvline(TRAIN_END, color='#6246ea', lw=1.2, ls='--', alpha=0.5)
ax.text(3, demand.max()*0.94, 'TRAIN', color='#0077b6', fontsize=9, fontweight='bold')
ax.text(TRAIN_END+2, demand.max()*0.94, 'TEST', color='#6246ea', fontsize=9, fontweight='bold')

ax.plot(np.arange(len(demand)), demand, color=C['demand'], lw=2, label='Actual Demand', zorder=5)
ax.plot(x[:len(f_naive)], f_naive, color=C['naive'], lw=1.3, ls=':', label='Naïve benchmark', alpha=0.8)
ax.plot(x[:len(f_ma3)],  f_ma3,  color=C['ma3'],  lw=1.8, ls='--', label='MA(3) — reactive')
ax.plot(x[:len(f_ma8)],  f_ma8,  color=C['ma8'],  lw=1.8, ls='--', label='MA(8) — balanced')
ax.plot(x[:len(f_ma12)], f_ma12, color=C['ma12'], lw=1.8, ls='--', label='MA(12) — smooth')
ax.annotate('Promo spike\n(Q3 event)', xy=(38, demand[38]),
            xytext=(43, demand[38]+500), fontsize=8, color='#c0392b',
            arrowprops=dict(arrowstyle='->', color='#c0392b'))
ax.set_title('Moving Average — FMCG Beverage Weekly Demand (104 weeks)', fontsize=13, fontweight='bold')
ax.set_ylabel('Weekly Demand (units)'); ax.legend(fontsize=9, loc='upper left')

ax2 = axes[1]
for f_arr, color, lbl in [(f_ma3,C['ma3'],'MA(3)'),(f_ma8,C['ma8'],'MA(8)'),(f_ma12,C['ma12'],'MA(12)')]:
    n_c = min(len(demand), len(f_arr))
    ax2.plot(f_arr[:n_c]-demand[:n_c], color=color, lw=1.2, alpha=0.8, label=lbl)
ax2.axhline(0, color='black', lw=0.8, ls='--')
ax2.axvline(TRAIN_END, color='#6246ea', lw=1.2, ls='--', alpha=0.5)
ax2.set_ylabel('Error (f-d)'); ax2.set_xlabel('Week'); ax2.legend(fontsize=8)
ax2.set_title('Residuals — patterns here = model missing something', fontsize=9)

plt.tight_layout()
plt.savefig('output_ch01_moving_average.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch01_moving_average.png')

In [ ]:
# ── 4. KPI TABLE ─────────────────────────────────────────────────────
results = {
    'Naïve': compute_kpis(demand, f_naive, f_naive, TRAIN_END),
    'MA(3)': compute_kpis(demand, f_ma3,   f_naive, TRAIN_END),
    'MA(8)': compute_kpis(demand, f_ma8,   f_naive, TRAIN_END),
    'MA(12)':compute_kpis(demand, f_ma12,  f_naive, TRAIN_END),
}
df_kpi = pd.DataFrame(results).T
print('=== KPI COMPARISON — TEST SET (weeks 81–104) ===')
print(df_kpi.to_string())
print()
print('BUSINESS ANSWER:')
print('  MA(8) gives the best balance for weekly FMCG replenishment.')
print('  MA(3) chases the promotional spike — inflates ordering costs.')
print('  MA(12) misses the summer ramp-up — 3-month lag causes stockouts.')
print('  Note: all MAs show negative FVA — Naïve wins because of strong trend.')
print('  This is the signal to move to Exponential Smoothing (Chapter 3).')

In [ ]:
# ── 5. LAG ANALYSIS ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
z0, z1 = 10, 55
xz = np.arange(z0, z1)

ax.fill_between(xz, demand[z0:z1], f_ma8[z0:z1],
                where=demand[z0:z1] > f_ma8[z0:z1],
                alpha=0.18, color='#c0392b', label='Under-forecast → stockout risk')
ax.fill_between(xz, demand[z0:z1], f_ma8[z0:z1],
                where=demand[z0:z1] < f_ma8[z0:z1],
                alpha=0.18, color='#e07b00', label='Over-forecast → overstock cost')
ax.plot(xz, demand[z0:z1], color=C['demand'], lw=2.2, label='Actual Demand')
ax.plot(xz, f_ma8[z0:z1],  color=C['ma8'],   lw=1.8, ls='--', label='MA(8) Forecast')

peak_d = np.argmax(demand[z0:z1]) + z0
peak_f = np.argmax(f_ma8[z0:z1])  + z0
lag = peak_f - peak_d
ax.annotate(f'Demand peak\nweek {peak_d}', xy=(peak_d, demand[peak_d]),
            xytext=(peak_d-7, demand[peak_d]+400),
            arrowprops=dict(arrowstyle='->', color=C['demand']), fontsize=9)
ax.annotate(f'MA(8) peak\nweek {peak_f} (lag={lag}w)', xy=(peak_f, f_ma8[peak_f]),
            xytext=(peak_f+2, f_ma8[peak_f]+400),
            arrowprops=dict(arrowstyle='->', color=C['ma8']), fontsize=9, color=C['ma8'])

ax.set_title('The Lag Problem — MA Always Follows, Never Leads (weeks 10–55 zoom)', fontsize=12, fontweight='bold')
ax.set_xlabel('Week'); ax.set_ylabel('Units'); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('output_ch01_lag_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch01_lag_analysis.png')

## Summary

| Finding | Business Implication |
|---------|---------------------|
| MA(8) has lowest MAE among MA variants | Best window for weekly FMCG replenishment |
| All MA models lag seasonal peak by 3–6 weeks | Stockout risk during summer ramp-up |
| Promotional spikes inflate MA(3) forecast | Erratic ordering → high operational cost |
| MA cannot beat Naïve on trending demand | Must upgrade to Exponential Smoothing |

**Next notebook:** `02_exponential_smoothing.ipynb` — solves flat-weighting via exponential decay